In [1]:
#new attempt at CNN on this dataset
#Alex Stedman
#we will be running a CNN on these spectrogram images to classify them
#use torchvision
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torchvision
from torchvision import datasets, transforms, models
from torch import nn, optim

#specify training and validation directories
train_dir = "../Datasets/SmellySongs9k/train"
val_dir = "../Datasets/SmellySongs9k/test"

#read out the dimensions of the images
example_image = plt.imread(os.path.join(train_dir, os.listdir(train_dir)[0], os.listdir(os.path.join(train_dir, os.listdir(train_dir)[0]))[0]))
img_height, img_width = example_image.shape[:2]
img_channels = example_image.shape[2] if len(example_image.shape) == 3 else 1
print(f"Image dimensions: {img_height}x{img_width}")
print(f"Image channels: {img_channels}")



Image dimensions: 224x224
Image channels: 4


In [27]:
#setup the datasets and dataloaders
batch_size = 32
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((img_height, img_width)),
        transforms.ToTensor(),
    ]),
    'val': transforms.Compose([
        transforms.Resize((img_height, img_width)),
        transforms.ToTensor(),
    ]),
}   
image_datasets = {
    'train': datasets.ImageFolder(train_dir, transform=data_transforms['train']),
    'val': datasets.ImageFolder(val_dir, transform=data_transforms['val']),
}
dataloaders = {
    'train': torch.utils.data.DataLoader(image_datasets['train'], batch_size=batch_size, shuffle=True),
    'val': torch.utils.data.DataLoader(image_datasets['val'], batch_size=batch_size, shuffle=False),
}
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
class_names = image_datasets['train'].classes
num_classes = len(class_names)
print(f"Classes: {class_names}")

#print out data stats
for phase in ['train', 'val']:
    print(f"{phase} dataset size: {dataset_sizes[phase]} images")


Classes: ['AI', 'Human']
train dataset size: 7950 images
val dataset size: 1962 images


In [28]:
#OPTIONAL SHRINK THE TRAINING AND VALIDATION DATASETS FOR TESTING. 
shrink_data = False
if shrink_data:
    small_train_size = 300
    small_val_size = 300
    image_datasets['train'], _ = torch.utils.data.random_split(image_datasets['train'], [small_train_size, dataset_sizes['train'] - small_train_size])
    image_datasets['val'], _ = torch.utils.data.random_split(image_datasets['val'], [small_val_size, dataset_sizes['val'] - small_val_size])
    dataloaders['train'] = torch.utils.data.DataLoader(image_datasets['train'], batch_size=batch_size, shuffle=True)
    dataloaders['val'] = torch.utils.data.DataLoader(image_datasets['val'], batch_size=batch_size, shuffle=False)
    dataset_sizes['train'] = small_train_size
    dataset_sizes['val'] = small_val_size
    print(f"Shrunk training dataset size: {dataset_sizes['train']} images")
    print(f"Shrunk validation dataset size: {dataset_sizes['val']} images")



In [29]:
#print out the class split in the datasets
print("Class distribution in training dataset:")
train_class_counts = np.zeros(num_classes, dtype=int)
for _, labels in dataloaders['train']:
    for label in labels:
        train_class_counts[label] += 1
for i, count in enumerate(train_class_counts):
    print(f"  {class_names[i]}: {count} images")
print("Class distribution in validation dataset:")
val_class_counts = np.zeros(num_classes, dtype=int)
for _, labels in dataloaders['val']:
    for label in labels:
        val_class_counts[label] += 1
for i, count in enumerate(val_class_counts):
    print(f"  {class_names[i]}: {count} images")


Class distribution in training dataset:
  AI: 3975 images
  Human: 3975 images
Class distribution in validation dataset:
  AI: 981 images
  Human: 981 images


In [30]:
#create a CNN for this classification task
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(),
            nn.Linear(128 * (img_height // 8) * (img_width // 8), 512),
            nn.ReLU(inplace=True),
            nn.Dropout(),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x


In [31]:
#initialize the model, loss function, and optimizer
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
#am I using the gpu?
print(f"Using device: {device}")
model = SimpleCNN(num_classes=num_classes)
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


Using device: cuda:1


In [32]:
#perform training
num_epochs = 20
for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")
    print("-" * 10)

    # Each epoch has a training and validation phase
    for phase in ['train', 'val']:
        if phase == 'train':
            model.train()  # Set model to training mode
        else:
            model.eval()   # Set model to evaluate mode

        running_loss = 0.0
        running_corrects = 0

        # Iterate over data.
        for inputs, labels in dataloaders[phase]:
            inputs = inputs.to(device)
            labels = labels.to(device)

            # zero the parameter gradients
            optimizer.zero_grad()

            # forward
            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                # backward + optimize only if in training phase
                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            # statistics
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        epoch_loss = running_loss / dataset_sizes[phase]
        epoch_acc = running_corrects.double() / dataset_sizes[phase]

        print(f"{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")

Epoch 1/20
----------
train Loss: 0.3440 Acc: 0.8297
val Loss: 0.0399 Acc: 0.9862
Epoch 2/20
----------
train Loss: 0.0318 Acc: 0.9881
val Loss: 0.0206 Acc: 0.9934
Epoch 3/20
----------
train Loss: 0.0076 Acc: 0.9976
val Loss: 0.0162 Acc: 0.9939
Epoch 4/20
----------
train Loss: 0.0089 Acc: 0.9971
val Loss: 0.0199 Acc: 0.9934
Epoch 5/20
----------
train Loss: 0.0031 Acc: 0.9990
val Loss: 0.0058 Acc: 0.9969
Epoch 6/20
----------
train Loss: 0.0091 Acc: 0.9971
val Loss: 0.0126 Acc: 0.9954
Epoch 7/20
----------
train Loss: 0.0027 Acc: 0.9991
val Loss: 0.0020 Acc: 0.9995
Epoch 8/20
----------
train Loss: 0.0021 Acc: 0.9994
val Loss: 0.0048 Acc: 0.9980
Epoch 9/20
----------
train Loss: 0.0034 Acc: 0.9989
val Loss: 0.0036 Acc: 0.9990
Epoch 10/20
----------
train Loss: 0.0006 Acc: 0.9997
val Loss: 0.0014 Acc: 0.9995
Epoch 11/20
----------
train Loss: 0.0030 Acc: 0.9990
val Loss: 0.0006 Acc: 1.0000
Epoch 12/20
----------
train Loss: 0.0002 Acc: 1.0000
val Loss: 0.0014 Acc: 0.9995
Epoch 13/20
-

In [33]:
#save the model
torch.save(model.state_dict(), "simple_cnn_songsniffer.pth")